# DurianGuard: Durian Leaf Disease Detection

This notebook builds an AI model that helps farmers identify durian leaf disease from a photo.

The workflow is written step by step so it is easy to explain in a competition presentation:

1. Get the dataset
2. Clean the image list
3. Explore the data with charts
4. Split images into train, validation, and test sets
5. Train several neural-network models
6. Evaluate and compare the models
7. Test one real image from the internet
8. Save outputs for the final report

Dataset: https://www.kaggle.com/datasets/cthng123/durian-leaf-disease-dataset?select=DLD_FinalDataset_224_spit

## 1. Install Required Libraries

This cell checks which Python packages are missing and installs only those packages. It works in Google Colab and local Jupyter.

In [ ]:
# Purpose: install missing helper libraries.
# You can run this cell safely more than once.

import importlib.util
import subprocess
import sys

required = ["kaggle", "PIL", "tqdm", "sklearn", "matplotlib", "seaborn", "pandas"]
pip_names = {"PIL": "Pillow", "sklearn": "scikit-learn"}
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    packages = [pip_names.get(pkg, pkg) for pkg in missing]
    print("Installing:", packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("All helper packages are already installed.")

## 2. Get the Dataset

Use this cell to download the Kaggle dataset. If automatic download does not work, download the zip file manually from Kaggle and extract it into the `data/` folder.

In [ ]:
# Purpose: find or download the Kaggle durian leaf disease dataset.
# If Kaggle download fails, manually extract the dataset into the data folder.

from pathlib import Path
import shutil
import subprocess
import sys

DATASET_SLUG = "cthng123/durian-leaf-disease-dataset"
DATA_ROOT = Path("data")
DATASET_DIR = DATA_ROOT / "DLD_FinalDataset_224_spit"

def maybe_upload_kaggle_token():
    try:
        import google.colab  # type: ignore
        from google.colab import files  # type: ignore
    except Exception:
        return

    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if kaggle_json.exists():
        return

    print("Upload kaggle.json from your Kaggle account settings.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        print("No kaggle.json uploaded. Manual dataset extraction is still possible.")
        return

    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.move("kaggle.json", kaggle_json)
    kaggle_json.chmod(0o600)
    print("Kaggle token saved to", kaggle_json)

def download_dataset_if_needed():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    known_dirs = [
        DATA_ROOT / "DLD_FinalDataset_224_spit",
        DATA_ROOT / "DLD_FinalDataset_224_split",
        DATA_ROOT / "DLD_FinalDataset_224",
    ]
    if any(path.exists() for path in known_dirs):
        print("Dataset already found under data/.")
        return

    maybe_upload_kaggle_token()
    subprocess.check_call([
        sys.executable,
        "-m",
        "kaggle",
        "datasets",
        "download",
        "-d",
        DATASET_SLUG,
        "-p",
        str(DATA_ROOT),
        "--unzip",
    ])

try:
    download_dataset_if_needed()
except Exception as exc:
    print("Automatic Kaggle download failed.")
    print("Manual download:", f"https://www.kaggle.com/datasets/{DATASET_SLUG}")
    print("Extract into:", DATA_ROOT.resolve())
    raise exc

for candidate in [
    DATA_ROOT / "DLD_FinalDataset_224_spit",
    DATA_ROOT / "DLD_FinalDataset_224_split",
    DATA_ROOT / "DLD_FinalDataset_224",
    DATA_ROOT,
]:
    if candidate.exists():
        DATASET_DIR = candidate
        break

print("Using dataset directory:", DATASET_DIR.resolve())

## 3. Experiment Settings

Most values you may want to change are inside `Config`. For a quick test, reduce `comparison_epochs`. For a serious final run, use GPU and increase epochs.

In [ ]:
# Purpose: import libraries and keep all experiment settings in one clear place.

from __future__ import annotations

from dataclasses import dataclass
import hashlib
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False

@dataclass
class Config:
    data_dir: Path = DATASET_DIR
    output_dir: Path = Path("outputs")
    reports_dir: Path = Path("reports")
    img_size: int = 224
    batch_size: int = 32
    comparison_epochs: int = 8
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    seed: int = 42
    val_split: float = 0.15
    test_split: float = 0.15
    patience: int = 5
    pretrained: bool = True
    freeze_backbone: bool = False
    use_class_weights: bool = True
    exclude_duplicate_images: bool = True
    use_existing_splits_when_available: bool = True
    num_workers: int = 0
    model_names_to_compare: tuple[str, ...] = (
        "efficientnet_b0",
        "mobilenet_v3_small",
        "resnet18",
    )

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg.reports_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(cfg)

## 4. Clean the Data

This section checks the image files and builds a clean table for training.

Important: it does not delete original images. It only creates clean CSV reports under `reports/`.

In [ ]:
# Purpose: create a clean image manifest without changing the original files.
# The clean manifest removes invalid images, exact duplicates, and label-conflict duplicates.

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {
    "train": "train",
    "training": "train",
    "val": "val",
    "valid": "val",
    "validation": "val",
    "test": "test",
    "testing": "test",
}

def file_md5(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Create a fingerprint for an image file so exact duplicates can be found."""
    digest = hashlib.md5()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def infer_label_and_split(path: Path, dataset_dir: Path):
    """Read the disease label and optional split name from the folder path."""
    parts = path.relative_to(dataset_dir).parts
    if len(parts) < 2:
        return None, None
    first = parts[0].lower()
    if first in SPLIT_ALIASES and len(parts) >= 3:
        return parts[1], SPLIT_ALIASES[first]
    return parts[0], None

def scan_image(path: Path, dataset_dir: Path):
    """Check one image and return its metadata for the manifest."""
    label, source_split = infer_label_and_split(path, dataset_dir)
    row = {
        "path": str(path.resolve()),
        "relative_path": str(path.relative_to(dataset_dir)),
        "label": label,
        "source_split": source_split,
        "extension": path.suffix.lower(),
        "file_size_kb": path.stat().st_size / 1024,
        "status": "ok",
        "error": "",
        "width": None,
        "height": None,
        "mode": None,
        "md5": None,
    }

    if label is None:
        row["status"] = "bad_layout"
        row["error"] = "Cannot infer label from folder structure"
        return row

    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            row["width"], row["height"] = image.size
            row["mode"] = image.mode
        row["md5"] = file_md5(path)
    except Exception as exc:
        row["status"] = "invalid_image"
        row["error"] = str(exc)
    return row

def build_clean_manifest(dataset_dir: Path):
    """Scan all images and return raw, clean, and summary tables."""
    image_paths = sorted(
        path
        for path in dataset_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No image files found under {dataset_dir.resolve()}")

    raw = pd.DataFrame(
        [scan_image(path, dataset_dir) for path in tqdm(image_paths, desc="Scanning images")]
    )
    raw.to_csv(cfg.reports_dir / "image_manifest_raw.csv", index=False)

    valid = raw[raw["status"].eq("ok")].copy()
    valid["label_conflict_duplicate"] = False
    valid["exact_duplicate"] = False

    if not valid.empty:
        label_counts = valid.groupby("md5")["label"].nunique()
        conflict_hashes = set(label_counts[label_counts > 1].index)
        valid["label_conflict_duplicate"] = valid["md5"].isin(conflict_hashes)
        valid["exact_duplicate"] = valid.duplicated("md5", keep="first")

    clean = valid[~valid["label_conflict_duplicate"]].copy()
    if cfg.exclude_duplicate_images:
        clean = clean[~clean["exact_duplicate"]].copy()

    clean = clean.reset_index(drop=True)
    clean.to_csv(cfg.reports_dir / "image_manifest_clean.csv", index=False)

    summary = {
        "total_files_scanned": int(len(raw)),
        "valid_images": int(len(valid)),
        "invalid_or_bad_layout": int(len(raw) - len(valid)),
        "exact_duplicate_rows": int(valid["exact_duplicate"].sum()) if len(valid) else 0,
        "label_conflict_duplicate_rows": int(valid["label_conflict_duplicate"].sum()) if len(valid) else 0,
        "clean_images_used": int(len(clean)),
        "classes": sorted(clean["label"].dropna().unique().tolist()),
    }
    with (cfg.reports_dir / "cleaning_summary.json").open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)
    return raw, clean, summary

raw_df, clean_df, cleaning_summary = build_clean_manifest(cfg.data_dir)
print(json.dumps(cleaning_summary, indent=2))
display(clean_df.head())

## 5. Explore the Dataset

EDA means exploratory data analysis. These charts show class balance, image sizes, and example leaf photos.

In [ ]:
# Purpose: show charts and sample images so we understand the dataset before training.

print("Raw image status")
display(raw_df["status"].value_counts().to_frame("count"))

print("Clean class count")
display(clean_df["label"].value_counts().sort_index().to_frame("count"))

plt.figure(figsize=(10, 4))
counts = clean_df["label"].value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values)
plt.xticks(rotation=25, ha="right")
plt.ylabel("image count")
plt.title("Clean dataset class distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
sns.histplot(clean_df["width"].dropna(), bins=20)
plt.title("Width")
plt.subplot(1, 3, 2)
sns.histplot(clean_df["height"].dropna(), bins=20)
plt.title("Height")
plt.subplot(1, 3, 3)
aspect = clean_df["width"].astype(float) / clean_df["height"].astype(float)
sns.histplot(aspect.dropna(), bins=20)
plt.title("Aspect ratio")
plt.tight_layout()
plt.show()

labels = sorted(clean_df["label"].unique())
samples_per_class = 3
plt.figure(figsize=(4 * samples_per_class, 3 * len(labels)))
plot_index = 1
for label in labels:
    sample = clean_df[clean_df["label"].eq(label)].sample(
        n=min(samples_per_class, (clean_df["label"] == label).sum()),
        random_state=cfg.seed,
    )
    for _, row in sample.iterrows():
        plt.subplot(len(labels), samples_per_class, plot_index)
        plt.imshow(Image.open(row["path"]).convert("RGB"))
        plt.title(label)
        plt.axis("off")
        plot_index += 1
plt.tight_layout()
plt.show()

## 6. Split Data Fairly

The model needs separate data for learning and testing:

- train: images used to learn
- validation: images used to choose the best model
- test: images used for final evaluation

If the dataset already has train/val/test folders, the notebook uses them. Otherwise it creates a new stratified split.

In [ ]:
# Purpose: create train, validation, and test splits for fair model evaluation.

def stratify_or_none(frame: pd.DataFrame):
    counts = frame["label"].value_counts()
    if counts.empty or counts.min() < 2:
        return None
    return frame["label"]

def make_splits(frame: pd.DataFrame):
    """Create train, validation, and test data with similar class balance."""
    frame = frame.copy().reset_index(drop=True)
    source_splits = set(frame["source_split"].dropna().unique())

    if cfg.use_existing_splits_when_available and {"train", "val"}.issubset(source_splits):
        frame["split"] = frame["source_split"]
        method = "existing train/val/test folders"
    elif cfg.use_existing_splits_when_available and {"train", "test"}.issubset(source_splits):
        train_source = frame[frame["source_split"].eq("train")].copy()
        test_source = frame[frame["source_split"].eq("test")].copy()
        train_part, val_part = train_test_split(
            train_source,
            test_size=cfg.val_split,
            random_state=cfg.seed,
            stratify=stratify_or_none(train_source),
        )
        train_part = train_part.copy()
        val_part = val_part.copy()
        test_source = test_source.copy()
        train_part["split"] = "train"
        val_part["split"] = "val"
        test_source["split"] = "test"
        frame = pd.concat([train_part, val_part, test_source], ignore_index=True)
        method = "existing train/test folders plus new validation split"
    else:
        holdout_size = cfg.val_split + cfg.test_split
        train_part, holdout = train_test_split(
            frame,
            test_size=holdout_size,
            random_state=cfg.seed,
            stratify=stratify_or_none(frame),
        )
        val_part, test_part = train_test_split(
            holdout,
            test_size=cfg.test_split / holdout_size,
            random_state=cfg.seed,
            stratify=stratify_or_none(holdout),
        )
        train_part = train_part.copy()
        val_part = val_part.copy()
        test_part = test_part.copy()
        train_part["split"] = "train"
        val_part["split"] = "val"
        test_part["split"] = "test"
        frame = pd.concat([train_part, val_part, test_part], ignore_index=True)
        method = "new stratified random split"

    class_names = sorted(frame["label"].unique().tolist())
    label_to_idx = {label: idx for idx, label in enumerate(class_names)}
    frame["label_idx"] = frame["label"].map(label_to_idx).astype(int)
    frame = frame.sample(frac=1.0, random_state=cfg.seed).reset_index(drop=True)
    frame.to_csv(cfg.reports_dir / "split_manifest.csv", index=False)
    return frame, class_names, label_to_idx, method

split_df, class_names, label_to_idx, split_method = make_splits(clean_df)
print("Split method:", split_method)
print("Classes:", class_names)
display(pd.crosstab(split_df["label"], split_df["split"], margins=True))

plt.figure(figsize=(10, 4))
sns.countplot(data=split_df, x="label", hue="split")
plt.xticks(rotation=25, ha="right")
plt.title("Class distribution by split")
plt.tight_layout()
plt.show()

## 7. Prepare Images for the Neural Network

This section turns the clean split table into PyTorch data loaders. It also applies image augmentation to training images so the model becomes more robust.

In [ ]:
# Purpose: convert image paths into PyTorch data loaders for training and testing.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def train_transform(img_size: int):
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.15),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.15, hue=0.03),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_transform(img_size: int):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class LeafDiseaseDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label_idx"]), row["path"]

def make_loader(frame: pd.DataFrame, transform, shuffle: bool):
    return DataLoader(
        LeafDiseaseDataset(frame, transform=transform),
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

set_seed(cfg.seed)
train_df = split_df[split_df["split"].eq("train")].copy()
val_df = split_df[split_df["split"].eq("val")].copy()
test_df = split_df[split_df["split"].eq("test")].copy()

if train_df.empty or val_df.empty:
    raise ValueError("Train and validation splits must not be empty.")

train_loader = make_loader(train_df, train_transform(cfg.img_size), shuffle=True)
val_loader = make_loader(val_df, eval_transform(cfg.img_size), shuffle=False)
test_loader = make_loader(test_df, eval_transform(cfg.img_size), shuffle=False) if not test_df.empty else None

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

In [ ]:
# Purpose: preview augmented training images after normalization is reversed for display.

def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1)

images, labels, paths = next(iter(train_loader))
count = min(8, len(images))
plt.figure(figsize=(14, 6))
for i in range(count):
    plt.subplot(2, 4, i + 1)
    plt.imshow(denormalize(images[i]).permute(1, 2, 0))
    plt.title(class_names[int(labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Helper Functions for Models and Evaluation

These functions keep the notebook organized. They build models, train one epoch, save checkpoints, collect predictions, and draw evaluation charts.

In [ ]:
# Purpose: define model creation, training, checkpoint, and metric helper functions.

SUPPORTED_MODELS = ["efficientnet_b0", "mobilenet_v3_small", "resnet18"]

def create_model(model_name: str, num_classes: int, pretrained: bool = True, freeze_backbone: bool = False):
    """Create a transfer-learning image classifier with a new output layer."""
    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        trainable_prefixes = ["classifier"]
    elif model_name == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        trainable_prefixes = ["classifier"]
    elif model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        trainable_prefixes = ["fc"]
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    if freeze_backbone:
        for name, parameter in model.named_parameters():
            parameter.requires_grad = any(name.startswith(prefix) for prefix in trainable_prefixes)
    return model

def model_parameter_count(model: nn.Module):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable

def make_class_weights(frame: pd.DataFrame):
    counts = frame["label_idx"].value_counts().reindex(range(len(class_names)), fill_value=0).sort_index()
    counts = counts.clip(lower=1).astype(float)
    weights = counts.sum() / (len(class_names) * counts)
    return torch.tensor(weights.values, dtype=torch.float32)

def run_epoch(model, loader, criterion, optimizer=None):
    """Run one training or evaluation pass and return key metrics."""
    is_train = optimizer is not None
    model.train(mode=is_train)
    running_loss = 0.0
    y_true, y_pred = [], []

    for images, labels, _paths in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(preds.detach().cpu().tolist())

    return {
        "loss": running_loss / max(len(loader.dataset), 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def save_checkpoint(path: Path, model: nn.Module, model_name: str, metrics: dict):
    torch.save({
        "model_name": model_name,
        "img_size": cfg.img_size,
        "class_names": class_names,
        "label_to_idx": label_to_idx,
        "model_state": model.state_dict(),
        "metrics": metrics,
    }, path)

def load_checkpoint(path: Path):
    checkpoint = torch.load(path, map_location=device)
    model = create_model(
        checkpoint["model_name"],
        num_classes=len(checkpoint["class_names"]),
        pretrained=False,
        freeze_backbone=False,
    ).to(device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, checkpoint

In [ ]:
# Purpose: collect predictions, evaluate model quality, and plot training curves.

@torch.no_grad()
def collect_predictions(model: nn.Module, loader):
    model.eval()
    rows = []
    for images, labels, paths in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        probabilities = torch.softmax(model(images), dim=1)
        scores, preds = torch.max(probabilities, dim=1)
        for actual_idx, pred_idx, score, path in zip(
            labels.tolist(),
            preds.cpu().tolist(),
            scores.cpu().tolist(),
            paths,
        ):
            rows.append({
                "path": path,
                "actual_idx": actual_idx,
                "pred_idx": pred_idx,
                "actual_label": class_names[actual_idx],
                "predicted_label": class_names[pred_idx],
                "confidence": score,
                "correct": actual_idx == pred_idx,
            })
    return pd.DataFrame(rows)

def evaluate_model(model: nn.Module, loader, split_name: str, output_dir: Path):
    """Create prediction CSV, classification report, and confusion matrix."""
    predictions = collect_predictions(model, loader)
    y_true = predictions["actual_idx"].tolist()
    y_pred = predictions["pred_idx"].tolist()

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(max(8, len(class_names) * 1.5), 6))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="YlGnBu", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=25, ha="right")
    plt.title(f"Confusion matrix: {split_name}")
    plt.tight_layout()
    matrix_path = output_dir / f"confusion_matrix_{split_name}.png"
    plt.savefig(matrix_path, dpi=180)
    plt.show()

    predictions.to_csv(output_dir / f"predictions_{split_name}.csv", index=False)
    with (output_dir / f"classification_report_{split_name}.json").open("w", encoding="utf-8") as file:
        json.dump(report, file, indent=2)

    return {
        f"{split_name}_accuracy": accuracy_score(y_true, y_pred),
        f"{split_name}_precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        f"{split_name}_recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        f"{split_name}_f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

@torch.no_grad()
def measure_inference_time_ms(model: nn.Module, repeats: int = 30):
    model.eval()
    sample = torch.randn(1, 3, cfg.img_size, cfg.img_size).to(device)
    for _ in range(5):
        _ = model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeats):
        _ = model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) * 1000 / repeats

def plot_history(history: list[dict], output_path: Path):
    frame = pd.DataFrame(history)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(frame["epoch"], frame["train_loss"], label="train")
    plt.plot(frame["epoch"], frame["val_loss"], label="validation")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(frame["epoch"], frame["train_accuracy"], label="train")
    plt.plot(frame["epoch"], frame["val_accuracy"], label="validation")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.show()

## 9. Train and Compare Models

This is the main experiment. The notebook trains each model listed in `cfg.model_names_to_compare`, then compares accuracy, macro F1, model size, and prediction speed.

In [ ]:
# Purpose: train one selected model and save its best checkpoint.

def train_one_model(model_name: str, epochs: int):
    """Train one model, evaluate it, and return one row for the comparison table."""
    set_seed(cfg.seed)
    run_dir = cfg.output_dir / model_name
    run_dir.mkdir(parents=True, exist_ok=True)

    model = create_model(
        model_name,
        num_classes=len(class_names),
        pretrained=cfg.pretrained,
        freeze_backbone=cfg.freeze_backbone,
    ).to(device)
    total_params, trainable_params = model_parameter_count(model)

    weights = make_class_weights(train_df).to(device) if cfg.use_class_weights else None
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))

    history = []
    best_val_f1 = -1.0
    epochs_without_improvement = 0
    started_at = time.time()
    print(f"Training {model_name} for up to {epochs} epochs")

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1_macro": train_metrics["f1_macro"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_precision_macro": val_metrics["precision_macro"],
            "val_recall_macro": val_metrics["recall_macro"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        print(
            f"Epoch {epoch:03d}/{epochs} "
            f"train_loss={row['train_loss']:.4f} train_acc={row['train_accuracy']:.4f} "
            f"val_loss={row['val_loss']:.4f} val_acc={row['val_accuracy']:.4f} val_f1={row['val_f1_macro']:.4f}"
        )

        if val_metrics["f1_macro"] > best_val_f1:
            best_val_f1 = val_metrics["f1_macro"]
            epochs_without_improvement = 0
            save_checkpoint(run_dir / "best_model.pt", model, model_name, {"best_val_f1_macro": best_val_f1, **val_metrics})
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.patience:
            print(f"Early stopping after {cfg.patience} epochs without improvement.")
            break

    save_checkpoint(run_dir / "last_model.pt", model, model_name, {"last_val_f1_macro": history[-1]["val_f1_macro"]})
    plot_history(history, run_dir / "training_curves.png")

    best_model, _checkpoint = load_checkpoint(run_dir / "best_model.pt")
    val_eval = evaluate_model(best_model, val_loader, "validation", run_dir)
    test_eval = evaluate_model(best_model, test_loader, "test", run_dir) if test_loader is not None else {}

    inference_ms = measure_inference_time_ms(best_model)
    checkpoint_mb = (run_dir / "best_model.pt").stat().st_size / (1024 * 1024)
    result = {
        "model_name": model_name,
        "epochs_ran": len(history),
        "best_val_f1_macro": best_val_f1,
        "validation_accuracy": val_eval["validation_accuracy"],
        "validation_precision_macro": val_eval["validation_precision_macro"],
        "validation_recall_macro": val_eval["validation_recall_macro"],
        "validation_f1_macro": val_eval["validation_f1_macro"],
        "test_accuracy": test_eval.get("test_accuracy", np.nan),
        "test_precision_macro": test_eval.get("test_precision_macro", np.nan),
        "test_recall_macro": test_eval.get("test_recall_macro", np.nan),
        "test_f1_macro": test_eval.get("test_f1_macro", np.nan),
        "total_params_million": total_params / 1_000_000,
        "trainable_params_million": trainable_params / 1_000_000,
        "checkpoint_mb": checkpoint_mb,
        "avg_inference_ms": inference_ms,
        "train_seconds": time.time() - started_at,
        "checkpoint_path": str(run_dir / "best_model.pt"),
    }
    with (run_dir / "metrics_summary.json").open("w", encoding="utf-8") as file:
        json.dump(result, file, indent=2)
    return result

### Run Model Comparison

This cell runs the experiment. It may take time, especially on CPU. For a quick test, set `cfg.comparison_epochs = 2` and compare only one model.

In [ ]:
# Purpose: run every model in the comparison list and choose the best checkpoint.

comparison_results = []
for model_name in cfg.model_names_to_compare:
    comparison_results.append(train_one_model(model_name, cfg.comparison_epochs))

comparison_df = pd.DataFrame(comparison_results).sort_values("validation_f1_macro", ascending=False).reset_index(drop=True)
comparison_df.to_csv(cfg.reports_dir / "model_comparison.csv", index=False)
display(comparison_df)

best_model_name = comparison_df.loc[0, "model_name"]
best_model_path = Path(comparison_df.loc[0, "checkpoint_path"])
print("Best model:", best_model_name)
print("Best checkpoint:", best_model_path)

## 10. Save Best Model for Streamlit Deployment

After comparing models, this section copies the winning checkpoint into a simple `deployment/` folder.

Streamlit should load this file:

```text
deployment/best_model.pt
```

The folder also stores `class_names.json` and `model_info.json`, which make the deployed app easier to understand and maintain.

In [ ]:
# Purpose: save the winning model in one stable deployment folder for Streamlit.

import shutil

deployment_dir = Path("deployment")
deployment_dir.mkdir(parents=True, exist_ok=True)

deploy_model_path = deployment_dir / "best_model.pt"
deploy_class_names_path = deployment_dir / "class_names.json"
deploy_model_info_path = deployment_dir / "model_info.json"

# Copy the best checkpoint chosen by the comparison table.
shutil.copy2(best_model_path, deploy_model_path)

# Save label order separately so the app/team can inspect it easily.
with deploy_class_names_path.open("w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=2)

# Save a small human-readable model card for deployment.
deployment_info = {
    "project": "DurianGuard",
    "task": "Durian leaf disease image classification",
    "best_model_name": best_model_name,
    "streamlit_model_path": str(deploy_model_path),
    "class_names": class_names,
    "selected_by": "highest validation macro F1 score in model comparison",
    "source_checkpoint": str(Path(comparison_df.loc[0, "checkpoint_path"])),
    "validation_f1_macro": float(comparison_df.loc[0, "validation_f1_macro"]),
    "test_f1_macro": None if pd.isna(comparison_df.loc[0, "test_f1_macro"]) else float(comparison_df.loc[0, "test_f1_macro"]),
    "checkpoint_mb": float(comparison_df.loc[0, "checkpoint_mb"]),
    "avg_inference_ms": float(comparison_df.loc[0, "avg_inference_ms"]),
}

with deploy_model_info_path.open("w", encoding="utf-8") as file:
    json.dump(deployment_info, file, indent=2)

# From this point on, prediction cells use the Streamlit-ready deployment model.
best_model_path = deploy_model_path

print("Deployment model saved:", deploy_model_path)
print("Class names saved:", deploy_class_names_path)
print("Model info saved:", deploy_model_info_path)
print("Use this path in Streamlit:", deploy_model_path)

## 11. Compare Results Visually

These charts help you explain which model is best for farmers. Higher F1 is better; smaller size and faster inference are better for deployment.

In [ ]:
# Purpose: compare model quality, size, and inference speed in one chart.

metric_columns = [
    "validation_accuracy",
    "validation_f1_macro",
    "test_accuracy",
    "test_f1_macro",
    "checkpoint_mb",
    "avg_inference_ms",
]
available_metrics = [column for column in metric_columns if column in comparison_df.columns]
comparison_long = comparison_df.melt(
    id_vars="model_name",
    value_vars=available_metrics,
    var_name="metric",
    value_name="value",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=comparison_long, x="metric", y="value", hue="model_name")
plt.xticks(rotation=25, ha="right")
plt.title("Model comparison")
plt.tight_layout()
plt.show()

display(comparison_df[[
    "model_name",
    "validation_f1_macro",
    "test_f1_macro",
    "checkpoint_mb",
    "avg_inference_ms",
    "checkpoint_path",
]])

## 12. Look at Model Mistakes

This section shows the most confident wrong predictions. These mistakes are useful because they tell us what kind of images the model still needs to learn.

In [ ]:
# Purpose: inspect wrong predictions so we know how to improve the model.

best_model, best_checkpoint = load_checkpoint(best_model_path)
analysis_loader = test_loader if test_loader is not None else val_loader
prediction_df = collect_predictions(best_model, analysis_loader)

print("Accuracy:", prediction_df["correct"].mean())
print("Most confident wrong predictions")
display(prediction_df[~prediction_df["correct"]].sort_values("confidence", ascending=False).head(10))

wrong = prediction_df[~prediction_df["correct"]].head(8)
if len(wrong) > 0:
    plt.figure(figsize=(14, 6))
    for i, (_, row) in enumerate(wrong.iterrows(), start=1):
        image = Image.open(row["path"]).convert("RGB")
        plt.subplot(2, 4, i)
        plt.imshow(image)
        plt.title(f"A: {row['actual_label']}\nP: {row['predicted_label']} ({row['confidence']:.2f})")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No wrong predictions in this split.")

## 13. Predict One Leaf Image

Use this section after training. Upload or set one image path, then the notebook predicts the disease class.

In [ ]:
# Purpose: upload or choose one image for prediction.

try:
    import google.colab  # type: ignore
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    IMAGE_PATH = Path(next(iter(uploaded.keys()))) if uploaded else None
except Exception:
    IMAGE_PATH = None

print("IMAGE_PATH =", IMAGE_PATH)

In [ ]:
# Purpose: predict the disease class for one leaf image.

@torch.no_grad()
def predict_image(image_path: Path, model_path: Path = best_model_path, top_k: int = 3):
    model, checkpoint = load_checkpoint(model_path)
    transform = eval_transform(int(checkpoint.get("img_size", cfg.img_size)))
    labels = checkpoint["class_names"]

    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)
    probabilities = torch.softmax(model(tensor), dim=1).squeeze(0)
    scores, indices = torch.topk(probabilities, k=min(top_k, len(labels)))

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

    results = []
    for score, index in zip(scores.cpu(), indices.cpu()):
        results.append({"class_name": labels[int(index)], "probability": float(score)})
    return results

# Example for local Jupyter:
# IMAGE_PATH = Path(r"C:\path\to\leaf.jpg")
if IMAGE_PATH:
    predictions = predict_image(Path(IMAGE_PATH), top_k=3)
    print(json.dumps(predictions, indent=2))
else:
    print("Set IMAGE_PATH to test a single image.")

## 14. Predict Internet Disease Image and Compare

This section downloads a labeled durian leaf disease image from the internet, saves it locally, predicts with the best trained model, and compares expected vs predicted.

The selected Roboflow sample is labeled **Leaf Blight**, so the expected label is `LEAF_BLIGHT`.

In [ ]:
# Purpose: download a labeled internet image and compare expected vs predicted class.

from urllib.request import urlretrieve

INTERNET_SAMPLE_URL = "https://source.roboflow.com/kqgFp9sbvAa3VE7fOf3MkkXTpoj2/0LTtQxJkZf52WNLWnqfq/thumb.jpg"
INTERNET_SAMPLE_SOURCE_PAGE = "https://universe.roboflow.com/dsi444/durian-disease"
EXPECTED_LABEL = "LEAF_BLIGHT"

sample_dir = Path("sample_images")
sample_dir.mkdir(parents=True, exist_ok=True)
internet_sample_path = sample_dir / "internet_leaf_blight_roboflow.jpg"

if not internet_sample_path.exists():
    urlretrieve(INTERNET_SAMPLE_URL, internet_sample_path)

print("Saved sample image:", internet_sample_path)
print("Source page:", INTERNET_SAMPLE_SOURCE_PAGE)
print("Expected label:", EXPECTED_LABEL)

image = Image.open(internet_sample_path).convert("RGB")
plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title(f"Internet sample - expected: {EXPECTED_LABEL}")
plt.show()

internet_predictions = predict_image(internet_sample_path, model_path=best_model_path, top_k=5)
prediction_table = pd.DataFrame(internet_predictions)
display(prediction_table)

def normalize_label(label: str) -> str:
    label = label.upper().replace("-", "_").replace(" ", "_")
    return "".join(ch for ch in label if ch.isalnum() or ch == "_")

top_prediction = internet_predictions[0]["class_name"]
match = normalize_label(top_prediction) == normalize_label(EXPECTED_LABEL)

comparison = {
    "expected_label": EXPECTED_LABEL,
    "predicted_label": top_prediction,
    "top_probability": internet_predictions[0]["probability"],
    "comparison_result": "MATCH" if match else "DIFFERENT",
}

print(json.dumps(comparison, indent=2))

if match:
    print("Result: model prediction matches the expected internet sample label.")
else:
    print("Result: model prediction is different from the expected internet sample label.")
    print("This can happen if the model has not trained enough, if labels differ between datasets, or if the internet image distribution is different from Kaggle.")

## 15. Competition Report Outputs

Use these generated files in your presentation and final report.

Deployment files for Streamlit:

- `deployment/best_model.pt`: best checkpoint copied to a stable app path
- `deployment/class_names.json`: label order used by the model
- `deployment/model_info.json`: small model card with metrics and deployment details